# Scaled dot product attention (SDPA)



I'm interested in fixing up the MPS implementation of SDPA in PyTorch. Currently, there is no dedicated backward function implementation for MPS, so I want to add one. In order to do that, I need to understand what SDPA does.

The SDPA operator was introduced in the well-known paper ["Attention Is All You Need"](https://papers.nips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf), with the following formula:

$$
\text{Attention}(Q, K, V)
= \text{softmax}\Biggl(
    \frac{Q K^T}{\sqrt{d_k}}
\Biggr) V
$$

Where:

* $Q$ and $K$ are matrices whose last dimension is $d_k$.
* $V$ is a matrix of dimension $d_v$.
* $ \text{softmax}(\mathbf x)_i = \dfrac{\exp(x_i)}{\sum_j \exp(x_j)} $, where $\mathbf x$ is a vector, is applied to the last dimension of the input.

The paper also mentions an additional optional step in which $Q K^T$ is masked so that particular elements contribute nothing to the softmax operation. So a more detailed formula would be:

$$
R
= \text{Attention}(Q, K, V)
= \text{softmax}\Biggl(
    \frac{\text{mask}(Q K^T, M)}{\sqrt{d_k}}
\Biggr) V
$$

With mask function

$$
\text{mask}(A, M)_{ij} = \begin{matrix}
A_{ij}, & \text{if } M_{ij} = \text{true} \\
-\infty, & \text{if } M_{ij} = \text{false}
\end{matrix}
$$

$M$ is a binary mask matrix of the same shape as $Q K^T$.

NOTE: Depending on implementation, this mask may sometimes be applied as a bias, where either 0 or $-\infty$ is added to each element of $Q K^T$. However, if one of the elements of $Q K^T$ is $\infty$, then the result becomes NaN. So it's not ideal to apply this operation as an added bias, but rather as a masked fill, or some other such operator that completely overrides the original value.

The PyTorch operator [`torch.nn.functional.scaled_dot_product_attention`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html) returns the output of $\text{Attention(Q, K, V)}$ as shown above. But it's worth mentioning that the `at::` implemetations also return a second value which is the intermediate result of the softmax operation, before the matrix multiplication with $V$, so that the backward call doesn't need to recalculate it when calculating the gradient for $V$.

NOTE: There is an additional "dropout" step where some elements of the softmax output are set to zero, either randomly with a probability value `dropout_p` or specifically with a mask `dropout_mask`. But neither the current MPS implementation nor the CPU implementation of the "flash" backend currently supports the dropout step, so I figure I can probably ignore it, at least for now.

There is also an optional `is_causal` flag. When enabled, it applies yet a different mask to $Q K^T$ which masks out all the upper right elements, so that only the lower left and diagonal elements remain. But if `is_causal=True` and an attention mask is also given, an error is raised, so we never apply both masks. So in the case of `is_causal=True`, we can just generate the following mask matrix given the matrix $A = Q K^T$:

$$
M(A)_{ij} = \begin{matrix}
A_{ij}, & \text{if } i \le j \\
-\infty, & \text{if } i > j
\end{matrix}
$$

There is also an optional `scale` input, which allows the user to override the $1/\sqrt{d_k}$ factor in the formula.

Finally, there's another optional flag called `enable_gqa`. I will account for it, but not yet.

The PyTorch documentation gives the following variable names and shapes for the inputs:

* $Q$: `query` $(..., L, E)$, where $E = d_k$ from above
* $K$: `key` $(..., S, E)$
* $V$: `value` $(..., S, E_v)$
* $M$: `attn_mask` $(..., L, S)$, optional
* $R$: output $(..., L, E_v)$

Below is my own implementation of SDPA with a test comparing its results with those of PyTorch, to make sure my understanding of the op is correct.

In [1]:
import torch

# NOTE: Not used at the moment
def softmax(a, dim):
    return torch.exp(a) / torch.exp(a).sum(dim=dim, keepdim=True)

def mask(a, attn_mask, is_causal):
    if attn_mask is not None or is_causal:
        if is_causal:
            assert attn_mask is None
            fill_mask = torch.ones_like(a, dtype=torch.bool).triu(diagonal=1)
        else:
            fill_mask = attn_mask.logical_not()

        a = a.masked_fill(fill_mask, -float('inf'))

    return a

def sdpa(query, key, value, attn_mask=None, is_causal=False, scale=None):
    if scale is None:
        scale = 1 / (query.size(-1))**0.5

    scaled_QK_T = torch.matmul(query, key.T) * scale
    masked_scaled_QK_T = mask(scaled_QK_T, attn_mask, is_causal)
    # NOTE: The above `softmax` func and `torch.softmax` do not give the
    # expected result when all the elements in a row are masked out, but
    # `torch._safe_softmax` does. I'd like to understand why.
    attn = torch._safe_softmax(masked_scaled_QK_T, dim=-1)
    return attn.matmul(value), attn

In [2]:
L = 10
E = 8
S = 5
E_v = 9

dtype = torch.double

mask_gens = [
    # lambda: (attn_mask, is_causal)
    lambda: (None, False),
    lambda: (torch.randn(L, S) > 0, False),
    lambda: (None, True),
]

for mask_gen in mask_gens:
    for _ in range(100):
        attn_mask, is_causal = mask_gen()

        query = torch.randn(L, E, dtype=dtype)
        key = torch.randn(S, E, dtype=dtype)
        value = torch.randn(S, E_v, dtype=dtype)

        # Check forward
        R, R_attn = sdpa(query, key, value, attn_mask=attn_mask, is_causal=is_causal)
        R_check = torch.nn.functional.scaled_dot_product_attention(query, key, value, attn_mask=attn_mask, is_causal=is_causal)
        assert torch.allclose(R, R_check)